In [3]:
from osgeo import gdal, osr
import numpy as np
import datetime
import netrc
from pathlib import Path
from shapely.geometry import box, MultiPolygon
import asf_search
import asf_search.constants.INTERNAL as asf_internal
import earthaccess
from tqdm import tqdm

asf_internal.CMR_TIMEOUT = 120

# ============================
# AUTHENTICATION (via ~/.netrc)
# ============================
# Requires a ~/.netrc entry:
#   machine urs.earthdata.nasa.gov
#   login your_username
#   password your_password
# chmod 600 ~/.netrc

auth = earthaccess.login(strategy='netrc')
endpoint = 'https://nisar.asf.earthdatacloud.nasa.gov/s3credentials'
s3_credentials = auth.get_s3_credentials(endpoint=endpoint)

netrc_login, _, netrc_password = netrc.netrc().authenticators('urs.earthdata.nasa.gov')
session = asf_search.ASFSession().auth_with_creds(netrc_login, netrc_password)

# ============================
# USER CONFIGURATION
# ============================
bbox = (-155.37, 19.28, -155.15, 19.48)  # (lon_min, lat_min, lon_max, lat_max)
start_date = "2025-10-21"   # or None
end_date   = "2026-07-05"   # or None
polarization = None         # e.g. "HH", "HV", "VV", "VH", or None
look_direction = None       # "ASCENDING", "DESCENDING", or None
outDir = "/home/matt/NISAR_cal_eval/GUNWS/Kilauea"
fileType = "GUNW"


def search_nisar(bbox, fileType, session,
                  start_date=None, end_date=None,
                  polarization=None):
    """
    Broad ASF search:
    - Filters ONLY by bbox, processing level, date range, optional polarization
    - Does NOT filter by look direction
    - Does NOT filter by orbit/frame
    - Does NOT filter by TF
    - Returns ALL granules intersecting the AOI
    """
    aoi_wkt = box(bbox[0], bbox[1], bbox[2], bbox[3]).wkt
    asf_search.constants.INTERNAL.CMR_TIMEOUT = 120

    opts = asf_search.ASFSearchOptions(
        collections=["C2850261892-ASF",    # OPERA-S1 GUNW
                     'C4052499921-ASF'],   # Provisional Bucket
        processingLevel="GUNW",
        intersectsWith=aoi_wkt,
        session=session
    )

    results = asf_search.geo_search(opts=opts)
    print(f"Found {len(results)} granules intersecting bounding box")

    def to_utc(dt_str):
        if dt_str is None:
            return None
        dt = datetime.datetime.fromisoformat(dt_str)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=datetime.timezone.utc)
        return dt

    start_dt = to_utc(start_date)
    end_dt   = to_utc(end_date)

    filtered = []
    for r in results:
        t = datetime.datetime.fromisoformat(r.properties["startTime"])
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)

        if start_dt and t < start_dt:
            continue
        if end_dt and t > end_dt:
            continue

        filtered.append(r)

    print(f"Granules after optional filters: {len(filtered)}")
    return filtered


def discover_TF(results):
    seen = {}
    for r in results:
        orbit = int(r.properties["pathNumber"])
        frame = int(r.properties["frameNumber"])
        pol   = r.properties["mainBandPolarization"]
        if isinstance(pol, list):
            pol = pol[0]
        key = (orbit, frame)
        if key not in seen:
            seen[key] = pol

    if not seen:
        print("❌ No TF combinations found")
        return None, None

    orbits = sorted(seen.keys())
    TF = np.array([[o, f] for o, f in orbits])
    poles = [seen[k] for k in orbits]
    return TF, poles


def get_GUNWs_date_range(
    TF, TFpol, session, projWin, mbox, interp,
    start_date, end_date, granules,
    outDir='/home/skg72/Documents/NISAR/Kaluapele',
):
    # ---- GDAL / VSICURL tuning ----
    gdal.SetConfigOption('CPL_VSIL_CURL_CHUNK_SIZE', '8388608')
    gdal.SetConfigOption('GDAL_HTTP_NETRC', 'YES')
    gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
    gdal.SetConfigOption('GDAL_CACHEMAX', '512')

    # Convert user inputs to timezone-aware UTC datetimes
    start_date = datetime.datetime.fromisoformat(start_date)
    end_date   = datetime.datetime.fromisoformat(end_date)

    if start_date.tzinfo is None:
        start_date = start_date.replace(tzinfo=datetime.timezone.utc)
    if end_date.tzinfo is None:
        end_date = end_date.replace(tzinfo=datetime.timezone.utc)

    pols    = ['HH', 'HV', 'VV', 'VH']
    polyIds = np.zeros((len(mbox.geoms)), dtype='object')
    allData = np.zeros((len(pols)), dtype='object')

    results = granules
    print(f"Total matching public granules found: {len(results)}")

    if len(results) == 0:
        print("❌ No granules found for this selection.")
        return [], [], [], [], []

    # Filter by date range
    filtered = []
    for r in results:
        t = datetime.datetime.fromisoformat(r.properties['startTime'])
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)
        if start_date <= t <= end_date:
            filtered.append(r)

    print(f"Granules within date range: {len(filtered)}")

    if len(filtered) == 0:
        print("❌ No granules fall within the requested date range.")
        return [], [], [], [], []

    # De-duplicate + sort by date
    resultList = []
    dates      = []
    crids      = []
    seen_files = set()

    for result in filtered:
        file_name = result.properties['fileName']
        if file_name in seen_files:
            continue
        seen_files.add(file_name)

        resultList.append(result)
        dates.append(datetime.datetime.fromisoformat(result.properties['startTime']))
        crids.append(result.properties.get('crid'))

    indices = np.argsort(dates)
    dates      = [dates[i]      for i in indices]
    resultList = [resultList[i] for i in indices]
    crids      = [crids[i]      for i in indices]

    print(f"Granules after de-duplication: {len(resultList)}")

    # Process remote granules
    outDir = Path(outDir)
    outDir.mkdir(parents=True, exist_ok=True)

    polStatus = np.zeros((len(resultList), len(pols)))
    starting  = np.ones(len(pols))

    geoTransforms = {}
    projections   = {}

    auth_session = auth.get_session()

    target_srs = osr.SpatialReference()
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(4326)

    print("\nProcessing granules:")
    for i, result in enumerate(tqdm(resultList, desc="Granules", unit="granule")):

        download_url = auth_session.get(
            result.properties['url'], allow_redirects=False
        ).headers['location']
        remote_path = '/vsicurl/' + download_url

        dataset = gdal.Open(remote_path, gdal.GA_ReadOnly)
        if dataset is None:
            print(f"Failed to open {remote_path}")
            continue

        subdatasets = dataset.GetSubDatasets()

        for j, pol in enumerate(pols):

            suffix = f'/science/LSAR/GUNW/grids/frequency{TF}/unwrappedInterferogram/{pol}/unwrappedPhase'

            matching_ds = next((ds_info[0] for ds_info in subdatasets if suffix in ds_info[0]), None)
            if matching_ds is None:
                continue

            polStatus[i, j] = 1

            mem_ds = gdal.Translate(
                '', matching_ds,
                format='MEM',
                projWin=projWin,
                projWinSRS='EPSG:4326',
                xRes=30, yRes=30,
                resampleAlg=interp
            )

            # Align onto the established reference grid so every granule's
            # array matches shape exactly, even if its native crop came out
            # a few pixels off from the reference.
            if not starting[j]:
                ref_gt = geoTransforms[j]
                ref_ny, ref_nx = allData[j].shape[1], allData[j].shape[2]
                mem_ds = gdal.Warp(
                    '', mem_ds,
                    format='MEM',
                    dstSRS=projections[j],
                    outputBounds=(
                        gdal.ApplyGeoTransform(ref_gt, 0, 0)[0],
                        gdal.ApplyGeoTransform(ref_gt, 0, ref_ny)[1],
                        gdal.ApplyGeoTransform(ref_gt, ref_nx, 0)[0],
                        gdal.ApplyGeoTransform(ref_gt, 0, 0)[1],
                    ),
                    width=ref_nx, height=ref_ny,
                    resampleAlg=interp
                )

            file_stem = Path(result.properties['fileName']).stem
            crop_name = f"{file_stem}_{pol}_unwrappedPhase.tif"
            crop_path = outDir / crop_name

            gdal.Translate(
                str(crop_path),
                mem_ds,
                format='GTiff',
                creationOptions=['COMPRESS=LZW']
            )

            tmp = mem_ds.GetRasterBand(1).ReadAsArray()

            if starting[j]:
                ny, nx = tmp.shape
                data = np.zeros((len(resultList), ny, nx), dtype='float32')

                geoTransforms[j] = mem_ds.GetGeoTransform()
                projections[j]   = mem_ds.GetProjection()

                inv_gt = gdal.InvGeoTransform(geoTransforms[j])
                target_srs.ImportFromWkt(projections[j])
                ct = osr.CoordinateTransformation(source_srs, target_srs)

                for k, geom in enumerate(mbox.geoms):
                    lonlat = list(geom.exterior.coords)
                    coords = []
                    for l in range(len(lonlat)):
                        utm_x, utm_y, _ = ct.TransformPoint(lonlat[l][1], lonlat[l][0])
                        coords.append(gdal.ApplyGeoTransform(inv_gt, utm_x, utm_y))
                    polyIds[k] = coords

                starting[j] = 0
            else:
                data = allData[j]

            data[i, :, :] = tmp.astype('float32')
            allData[j]    = data

        dataset = None

    return dates, polStatus, allData, polyIds, crids


def run_nisar_pipeline(
    bbox,
    fileType,
    session,
    outDir,
    start_date=start_date,
    end_date=end_date,
    polarization=polarization,
    look_direction=look_direction
):
    # 1. Search ASF for GUNW granules
    results = search_nisar(
        bbox=bbox,
        fileType=fileType,
        session=session,
        start_date=start_date,
        end_date=end_date,
        polarization=None      # GUNW is always VV
    )
    if len(results) == 0:
        print("❌ No GUNW granules found after filtering")
        return

    min_lon, min_lat, max_lon, max_lat = bbox
    mbox = MultiPolygon([box(min_lon, min_lat, max_lon, max_lat)])

    print(f"\nProcessing {len(results)} GUNW granules…")

    # 2. Process ALL granules at once (no TF grouping)
    get_GUNWs_date_range(
        TF='A',
        TFpol="VV",
        session=session,
        projWin=[min_lon, max_lat, max_lon, min_lat],
        mbox=mbox,
        interp="bilinear",
        start_date=start_date,
        end_date=end_date,
        outDir=outDir,
        granules=results
    )

    print("\n🎉 Pipeline complete — all GUNW crops saved.")


run_nisar_pipeline(
    bbox=bbox,
    fileType=fileType,
    session=session,
    outDir=outDir,
    start_date=start_date,
    end_date=end_date,
    polarization=polarization,
    look_direction=look_direction
)

Found 32 granules intersecting bounding box
Granules after optional filters: 32

Processing 32 GUNW granules…
Total matching public granules found: 32
Granules within date range: 32
Granules after de-duplication: 32

Processing granules:


Granules: 100%|██████████| 32/32 [05:48<00:00, 10.90s/granule]


🎉 Pipeline complete — all GUNW crops saved.


In [3]:
import asf_search as asf_search

# ─────────────────────────────────────────────
# Query CMR for all collections with "NISAR" in the name
# ─────────────────────────────────────────────
opts = asf_search.ASFSearchOptions(
    provider='ASF',
)

# ASF's collection search - short_name search across CMR
import requests

def find_nisar_collections():
    """
    Queries NASA CMR directly for all collections whose short_name
    or entry_title mentions NISAR. Returns concept_id (the collection ID
    used with `collections=[...]` in asf_search.ASFSearchOptions),
    short_name, and version/status info.
    """
    url = "https://cmr.earthdata.nasa.gov/search/collections.json"
    params = {
        "keyword": "NISAR",
        "provider": "ASF",
        "page_size": 100,
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    entries = resp.json()["feed"]["entry"]

    collections = []
    for e in entries:
        collections.append({
            "concept_id": e.get("id"),
            "short_name": e.get("short_name"),
            "version_id": e.get("version_id"),
            "title": e.get("title"),
        })
    return collections


if __name__ == "__main__":
    collections = find_nisar_collections()
    print(f"Found {len(collections)} NISAR-related collections:\n")
    for c in collections:
        print(f"{c['concept_id']:20s}  {c['short_name']:30s}  v{c['version_id']:5s}  {c['title']}")

Found 80 NISAR-related collections:

C3622214170-ASF       NISAR_L2_GCOV_BETA_V1           v1      NISAR Beta Geocoded Polarimetric Covariance Product (Version 1)
C2850225585-ASF       NISAR_L1_RSLC_BETA_V1           v1      NISAR Beta Range Doppler Single Look Complex Product (Version 1)
C2850259510-ASF       NISAR_L2_GSLC_BETA_V1           v1      NISAR Beta Geocoded Single Look Complex Product (Version 1)
C2850261892-ASF       NISAR_L2_GUNW_BETA_V1           v1      NISAR Beta Geocoded Unwrapped Interferogram Product (Version 1)
C2850265000-ASF       NISAR_L3_SME2_BETA_V1           v1      NISAR Beta Soil Moisture (Version 1)
C2850263910-ASF       NISAR_L2_GOFF_BETA_V1           v1      NISAR Beta Geocoded Pixel Offsets (Version 1)
C2850237619-ASF       NISAR_L1_ROFF_BETA_V1           v1      NISAR Beta Range Doppler Pixel Offsets (Version 1)
C2850235455-ASF       NISAR_L1_RUNW_BETA_V1           v1      NISAR Beta Range Doppler Unwrapped Interferogram Product (Version 1)
C4052499921